In [12]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
import os
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [13]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)

    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)

    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns to be safe (besides index)
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        print(f"Warning: Found unexpected object columns: {object_cols}. Dropping them.")
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)

    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train


In [14]:
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)


0

In [17]:
# =============================================================================
# BLOCK: TUNE XGBOOST MEAN MODEL (NATIVE API - CORRECTED)
# =============================================================================
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import optuna
import json
import numpy as np

# --- 1. Prepare Data for Tuning ---
# This step remains the same.
print("--- Step 1: Preparing data for tuning... ---")
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X, y_true, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Data prepared. Training set size: {len(X_train_opt)}, Validation set size: {len(X_val_opt)}")

# --- 2. Define the Optuna Objective Function using Native XGBoost ---
def objective_xgboost_native(trial):
    """
    This function uses the native xgb.train API, which requires xgb.DMatrix objects.
    """
    # Create the DMatrix objects required by the native API
    dtrain = xgb.DMatrix(X_train_opt, label=y_train_opt)
    dval = xgb.DMatrix(X_val_opt, label=y_val_opt)

    # Define the hyperparameter search space (this is compatible with the native API)
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'n_jobs': -1,
        'seed': RANDOM_STATE,
        'eta': trial.suggest_float('eta', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'lambda': trial.suggest_float('lambda', 1e-4, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True)
    }

    # === THE FIX IS HERE: Using xgb.train with its stable parameters ===
    bst = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=2500,  # Max number of boosting rounds
        evals=[(dval, 'validation')],
        early_stopping_rounds=100,
        verbose_eval=False  # Suppress verbose output during training
    )

    # Make predictions on the validation set using the best iteration
    preds = bst.predict(dval, iteration_range=(0, bst.best_iteration))
    rmse = np.sqrt(mean_squared_error(y_val_opt, preds))
    
    # Store the best number of rounds for later use
    trial.set_user_attr("best_iteration", bst.best_iteration)
    
    print(f"  Trial {trial.number}: RMSE = ${rmse:,.2f}")
    return rmse

# --- 3. Create and Run the Optuna Study ---
study_xgboost = optuna.create_study(direction='minimize')
print("\n--- Step 2: Starting XGBoost Hyperparameter Tuning (Native API)... ---")
study_xgboost.optimize(objective_xgboost_native, n_trials=N_OPTUNA_TRIALS)

# --- 4. Print the Best Results ---
best_params_xgb = study_xgboost.best_params
print("\n--- Step 3: XGBoost Tuning Complete ---")
print(f"Best trial validation RMSE: ${study_xgboost.best_value:,.2f}")
print("Best hyperparameters found:")
print(best_params_xgb)

# --- 5. Retrain on Full Data and Save Artifacts ---
print("\n--- Step 4: Retraining final XGBoost model on all data (Native API)... ---")
# Get the optimal number of boosting rounds from the best trial
best_n_estimators = study_xgboost.best_trial.user_attrs['best_iteration']
print(f"Using {best_n_estimators} boosting rounds for the final model.")

# Create a DMatrix for the ENTIRE training dataset
dtrain_full = xgb.DMatrix(X, label=y_true)

# Train the final model on all data
final_bst = xgb.train(
    params=best_params_xgb,
    dtrain=dtrain_full,
    num_boost_round=best_n_estimators,
    verbose_eval=False
)



[I 2025-07-22 16:05:01,615] A new study created in memory with name: no-name-19427d7c-d7cd-45d6-ad31-20dd77f8416a


--- Step 1: Preparing data for tuning... ---
Data prepared. Training set size: 160000, Validation set size: 40000

--- Step 2: Starting XGBoost Hyperparameter Tuning (Native API)... ---


[I 2025-07-22 16:05:44,627] Trial 0 finished with value: 100088.05595074769 and parameters: {'eta': 0.04990674371699128, 'max_depth': 5, 'subsample': 0.9319112441734054, 'colsample_bytree': 0.7124102172228501, 'lambda': 0.2924071513433339, 'alpha': 0.00164251234561687}. Best is trial 0 with value: 100088.05595074769.


  Trial 0: RMSE = $100,088.06


[I 2025-07-22 16:07:47,736] Trial 1 finished with value: 99927.34640727732 and parameters: {'eta': 0.03156384528890795, 'max_depth': 10, 'subsample': 0.9358400910262724, 'colsample_bytree': 0.9172699776643372, 'lambda': 0.6240368672231683, 'alpha': 0.0025190920015068733}. Best is trial 1 with value: 99927.34640727732.


...

  Trial 27: RMSE = $100,066.23


[I 2025-07-22 16:39:16,059] Trial 28 finished with value: 98262.42349952499 and parameters: {'eta': 0.049445777569295676, 'max_depth': 8, 'subsample': 0.9917901681531063, 'colsample_bytree': 0.6111321755244464, 'lambda': 0.004280720805557039, 'alpha': 0.005309519032526359}. Best is trial 23 with value: 97725.2023175189.


  Trial 28: RMSE = $98,262.42


[I 2025-07-22 16:40:00,022] Trial 29 finished with value: 98695.0866456887 and parameters: {'eta': 0.044818125116662985, 'max_depth': 6, 'subsample': 0.9561452963294016, 'colsample_bytree': 0.730626672949457, 'lambda': 0.11935123971713416, 'alpha': 0.0013037953709835238}. Best is trial 23 with value: 97725.2023175189.


  Trial 29: RMSE = $98,695.09

--- Step 3: XGBoost Tuning Complete ---
Best trial validation RMSE: $97,725.20
Best hyperparameters found:
{'eta': 0.026187265423100955, 'max_depth': 8, 'subsample': 0.8698093182196516, 'colsample_bytree': 0.7346009666347628, 'lambda': 0.6781225709954017, 'alpha': 3.2533324302845594}

--- Step 4: Retraining final XGBoost model on all data (Native API)... ---
Using 2498 boosting rounds for the final model.


In [ ]:
# =============================================================================
# BLOCK: RETRAIN AND SAVE FINAL XGBOOST MODEL (FROM OPTUNA RESULTS)
# =============================================================================
import xgboost as xgb
import json

# --- Step 4 (Redux): Retraining final XGBoost model on all data ---
print("--- Picking up from Step 4: Retraining final XGBoost model on all data... ---")

# Manually define the best parameters found in the previous run.
# These values are taken directly from your output log.
best_params_xgb = {
    'eta': 0.026187265423100955, 
    'max_depth': 8, 
    'subsample': 0.8698093182196516, 
    'colsample_bytree': 0.7346009666347628, 
    'lambda': 0.6781225709954017, 
    'alpha': 3.2533324302845594,
    
    # Add the fixed parameters back in for the training function
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_jobs': -1,
    'seed': RANDOM_STATE
}
best_n_estimators = 2498 

print("\nUsing the following best parameters found from Optuna:")
print(f"  - Parameters: {best_params_xgb}")
print(f"  - Boosting Rounds: {best_n_estimators}")

# Create a DMatrix for the ENTIRE training dataset
print("\nCreating DMatrix for the full training data... (this is memory-intensive)")
dtrain_full = xgb.DMatrix(X, label=y_true)

# Train the final model on all data
print("Training the final model... (this may take a moment)")
final_bst = xgb.train(
    params=best_params_xgb,
    dtrain=dtrain_full,
    num_boost_round=best_n_estimators,
    verbose_eval=False  # No need for verbose output here
)
print("Final model training complete.")

# --- Step 5: Save the final model and parameters to files ---
print("\n--- Step 5: Saving the final model and parameters... ---")
model_filename = 'xgboost_mean_model.json'
params_filename = 'xgboost_best_params.json'

# The native 'save_model' method is the correct way to save a native booster object.
final_bst.save_model(model_filename)

# Save the raw Optuna parameters without the fixed keys for clean reuse later
optuna_best_params = study_xgboost.best_params
with open(params_filename, 'w') as f:
    json.dump(optuna_best_params, f)

print(f"\nFinal XGBoost model saved to '{model_filename}'")
print(f"Best parameters saved to '{params_filename}'")

--- Picking up from Step 4: Retraining final XGBoost model on all data... ---

Using the following best parameters found from Optuna:
  - Parameters: {'eta': 0.026187265423100955, 'max_depth': 8, 'subsample': 0.8698093182196516, 'colsample_bytree': 0.7346009666347628, 'lambda': 0.6781225709954017, 'alpha': 3.2533324302845594, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'n_jobs': -1, 'seed': 42}
  - Boosting Rounds: 2498

Creating DMatrix for the full training data... (this is memory-intensive)
Training the final model... (this may take a moment)


In [ ]:
import os
import json

# --- Step 5 (Revised): Saving the final model and parameters to a specific folder ---
print("\n--- Step 5: Saving the final model and parameters... ---")

# Define the directory path
save_dir = 'mean_models'

# Create the directory if it does not exist. 
# 'exist_ok=True' prevents an error if the folder is already there.
os.makedirs(save_dir, exist_ok=True)
print(f"Ensured directory '{save_dir}' exists.")

# Define the full path for each file
model_filename = os.path.join(save_dir, 'xgboost_mean_model.json')
params_filename = os.path.join(save_dir, 'xgboost_best_params.json')

# The native 'save_model' method is the correct way to save a native booster object.
final_bst.save_model(model_filename)

# Save the raw Optuna parameters without the fixed keys for clean reuse later
optuna_best_params = study_xgboost.best_params # Assumes study_xgboost is still in memory
with open(params_filename, 'w') as f:
    json.dump(optuna_best_params, f)

print(f"\nFinal XGBoost model saved to: '{model_filename}'")
print(f"Best parameters saved to: '{params_filename}'")